# QuickCart Warehouse Inventory Stockout Risk

**Goal:** classify each store-SKU-day as **Safe, At-Risk or Imminent**.

This notebook accompanies `quickcart_stockout_risk.py` and the final report.

In [ ]:
import pandas as pd
from pathlib import Path
BASE=Path('.')
stores=pd.read_csv(BASE/'dim_stores.csv')
skus=pd.read_csv(BASE/'dim_skus.csv')
suppliers=pd.read_csv(BASE/'dim_suppliers.csv',na_values=['N/A','missing','--','NA','null'],keep_default_na=True)
events=pd.read_csv(BASE/'dim_events.csv',parse_dates=['date'])
fact=pd.read_csv(BASE/'fact_inventory_daily.csv',parse_dates=['date'])
print(len(stores),len(skus),len(suppliers),len(events),len(fact))

In [ ]:
print(fact['stockout_risk'].value_counts())
print('Missing actual lead time:',fact['lead_time_days_actual'].isna().sum())
print('Supplier reliability missing:',suppliers['reliability_score'].isna().sum())
print('City display mismatches:')
print(stores.loc[stores.city_display!=stores.city,['store_id','city','city_display']])

## Modeling workflow
1. Join the five tables.
2. Clean supplier reliability (`N/A` → missing → median imputation).
3. Engineer reorder gap, normalized cover, recent reorder and festival features.
4. Use a leakage-aware temporal split: Oct 1–23 train, Oct 24–30 test.
5. Compare baseline, Logistic Regression and Random Forest.
6. Tune the Imminent threshold only on Oct 19–23 validation data.

In [ ]:
# Run the supplied quickcart_stockout_risk.py script for the full reproducible pipeline.

## Completed results

- **Random Forest test accuracy:** 94.66%
- **Balanced accuracy:** 90.36%
- **Imminent recall:** 82.97%
- **Imminent precision:** 82.97%
- **Macro F1:** 90.32%

The 0.35 Imminent probability threshold is a practical recall/precision compromise selected using an internal temporal validation window.